In [ ]:
# from typing import Literal

# import pydantic

# from theia.types import (
#     KillEvent,
#     MonostaticRadarDetection,
#     PclDetection,
#     Shot,
#     Snapshot,
#     TrackInitEvent,
# )


# class LogFileDetections(pydantic.BaseModel):
#     team: Literal["blue", "red"]
#     active_radar_detections: list[MonostaticRadarDetection]
#     pcl_detections: list[PclDetection]


# class LogFile(pydantic.BaseModel):
#     snapshots: list[Snapshot]
#     detections: list[LogFileDetections]
#     events: list[TrackInitEvent | Shot | KillEvent]


# with open("result_full_sensors.json", "r") as file:
#     f = LogFile.model_validate_json(file.read())

In [ ]:
from theia.simulation.analysis import Analysis
from theia.simulation.theia_logging import LogLoader

loader = LogLoader("result_full_sensors.json")
analysis = Analysis(log_file=loader)

In [ ]:
from theia.simulation.scenario_import import ScenarioFactory

with open("full_sensors.json", "r") as file:
    scenario = ScenarioFactory.model_validate_json(file.read())

In [ ]:
n_shots = analysis.n_shots_per_effector
n_shots

In [ ]:
import folium

from theia.coordinates import CoordinateTransformations
from theia.coverage import calculate_coverage


m = folium.Map((47.45270, 8.56068), zoom_start=10)
folium.LatLngPopup().add_to(m)
for e in scenario.blue_orbat.effectors:
    n = n_shots[e.effector.id]
    l = (e.effector.point.lat, e.effector.point.lon)
    color = "gray" if n == 0 else "blue"
    folium.Marker(
        location=l,
        tooltip=f"Effector ID {e.effector.id}\n #shots = {n}",
        icon=folium.Icon(color=color),
    ).add_to(m)
    folium.GeoJson(
        calculate_coverage(
            scenario.terrain_model.to_terrain(),
            e.effector.point,
            e.effector.combat_range,
            1000,
        ),
        color=color,
    ).add_to(m)
    # folium.Circle(location=l, radius=e.effector.combat_range, color=color).add_to(m)
for target_id, t in loader.red_target_ground_truth.items():
    points = [
        CoordinateTransformations.cartesian_to_geodetic(*state.state_vector[[0, 2, 4]])[
            :2
        ]
        for state in t.states
    ]
    folium.PolyLine(
        points,
        tooltip=f"Target ID #{target_id}",
        color="gray" if target_id in analysis.times_of_death else "red",
    ).add_to(m)

m